# INC-007 | Acesso da conta da aplicação

## Relato do cliente
"A TI suspeita que a conta usada pela aplicação tenha permissões além das necessárias para as rotinas do sistema. Precisamos identificar quais acessos são necessários e quais podem ser revistos."

## Hipótese inicial
A conta de banco de dados `pbl_app`, utilizada pelo serviço de aplicação, foi configurada com privilégios excessivos e irrestritos no banco de dados (`ALL PRIVILEGES` e `CREATE ON SCHEMA`), violando o **Princípio do Menor Privilégio (PoLP)**. Essa configuração permite que a aplicação execute operações DDL destrutivas (`DROP TABLE`, `CREATE TABLE`) e operações DML de alto risco (`TRUNCATE`, `DELETE` irrestrito em históricos e auditoria), expondo o ambiente produtivo a riscos severos de corrupção ou indisponibilidade de dados.

In [ ]:
from pathlib import Path
from dotenv import dotenv_values
import psycopg
import pandas as pd

raiz = Path.cwd().resolve()
if raiz.name == "notebooks":
    raiz = raiz.parent

config = dotenv_values(raiz / "docker" / ".env")

def conectar_testes():
    return psycopg.connect(
        host="127.0.0.1",
        port=55432,
        dbname="pbl_industria_testes",
        user="aluno",
        password=config.get("POSTGRES_PASSWORD", "pipe101198")
    )

## 1. Auditoria da concessão de privilégios legada

O arquivo de configuração `configuracao/00_configurar_acesso.sql` (linhas 16 a 19) define o perfil de segurança concedido ao usuário `pbl_app`:

```sql
SELECT format('GRANT CONNECT ON DATABASE %I TO pbl_app;', current_database()) \gexec
GRANT USAGE, CREATE ON SCHEMA industria TO pbl_app;
GRANT ALL PRIVILEGES ON ALL TABLES IN SCHEMA industria TO pbl_app;
GRANT ALL PRIVILEGES ON ALL SEQUENCES IN SCHEMA industria TO pbl_app;
```

### Privilégios perigosos identificados:
- `CREATE ON SCHEMA industria`: Concede capacidade DDL para criar e substituir objetos dentro do schema do negócio.
- `ALL PRIVILEGES ON ALL TABLES`: Inclui `TRUNCATE`, `REFERENCES`, `TRIGGER` e `DELETE`.
- Falta de granularidade: A aplicação não deve possuir permissão de exclusão (`DELETE` ou `TRUNCATE`) sobre tabelas de histórico ou trilhas de auditoria (`movimentacoes`, `auditoria`, `itens_ordem`).

In [ ]:
# Consulta dos privilégios efetivos concedidos à role pbl_app
sql_privilegios = """
SELECT
    table_name AS tabela,
    privilege_type AS privilegio,
    is_grantable
FROM information_schema.table_privileges
WHERE grantee = 'pbl_app'
  AND table_schema = 'industria'
ORDER BY table_name, privilege_type;
"""

with conectar_testes() as conn:
    df_privilegios = pd.read_sql(sql_privilegios, conn)

print("Amostra dos privilégios concedidos ao pbl_app:")
display(df_privilegios.head(25))

# Quantificar os tipos de privilégios concedidos
display(df_privilegios["privilegio"].value_counts().to_frame("Total de Tabelas"))

## 2. Demonstração prática do risco de segurança

Com as permissões atuais, uma falha na aplicação, injeção de SQL ou comando acidental pode executar comandos de destruição maciça, como o truncamento da tabela de movimentações de 500 mil registros:

In [ ]:
# Demonstração: pbl_app consegue executar TRUNCATE ou criar tabelas DDL
def conectar_app():
    return psycopg.connect(
        host="127.0.0.1",
        port=55432,
        dbname="pbl_industria_testes",
        user="pbl_app",
        password="SENHA_LOCAL_DO_PBL_APP" # Utiliza a conta da aplicação
    )

# Verificação se o pbl_app possui permissão de DDL (CREATE TABLE)
try:
    with conectar_app() as conn:
        with conn.cursor() as cur:
            cur.execute("CREATE TABLE industria.tabela_teste_permissao (id int);")
            cur.execute("DROP TABLE industria.tabela_teste_permissao;")
            conn.commit()
            print("VULNERABILIDADE COMPROVADA: A conta da aplicação conseguiu criar e apagar tabelas DDL!")
except Exception as e:
    print(f"Comando barrado: {e}")

## 3. Matriz de permissões necessárias vs privilégios revistos

Mapeamos os acessos estritamente necessários para as rotinas operacionais:

| Tabela | Privilégios Necessários | Privilégios a Revogar | Justificativa |
|:---|:---:|:---:|:---|
| `auditoria` | `SELECT, INSERT` | `UPDATE, DELETE, TRUNCATE` | Trilha de auditoria deve ser estritamente append-only / imutável |
| `movimentacoes` | `SELECT, INSERT` | `UPDATE, DELETE, TRUNCATE` | Histórico financeiro e físico não pode sofrer exclusões arbitrárias |
| `estoques` | `SELECT, INSERT, UPDATE` | `DELETE, TRUNCATE` | Operação do almoxarifado não remove depósitos de itens |
| `componentes` | `SELECT, UPDATE` | `INSERT, DELETE, TRUNCATE` | Atualização de preços de catálogo; criação de itens cabe a cadastro |
| `depositos` | `SELECT` | `INSERT, UPDATE, DELETE, TRUNCATE` | Apenas consulta nas operações da aplicação |
| `equipamentos` | `SELECT` | `INSERT, UPDATE, DELETE, TRUNCATE` | Apenas consulta nas operações da aplicação |
| `fornecedores` | `SELECT` | `INSERT, UPDATE, DELETE, TRUNCATE` | Apenas consulta nas fichas |
| `importacoes` | `SELECT, INSERT, UPDATE` | `DELETE, TRUNCATE` | Registro de lotes e conclusão de status |
| `itens_ordem` | `SELECT, INSERT` | `UPDATE, DELETE, TRUNCATE` | Itens agregados à ordem de serviço |
| `ordens_servico` | `SELECT, INSERT, UPDATE` | `DELETE, TRUNCATE` | Abertura, mudança de status e encerramento |
| `servicos_ordem` | `SELECT, INSERT` | `UPDATE, DELETE, TRUNCATE` | Registro de serviços prestados |
| `tecnicos` | `SELECT` | `INSERT, UPDATE, DELETE, TRUNCATE` | Cadastro de técnicos |
| **Schema `industria`** | `USAGE` | `CREATE` | Impede qualquer comando DDL pela aplicação |

In [ ]:
# Script SQL para implementação do Princípio do Menor Privilégio
sql_revisao_acessos = """
-- 1. Revogar permissões perigosas
REVOKE CREATE ON SCHEMA industria FROM pbl_app;
REVOKE ALL PRIVILEGES ON ALL TABLES IN SCHEMA industria FROM pbl_app;

-- 2. Conceder uso do schema e sequências
GRANT USAGE ON SCHEMA industria TO pbl_app;
GRANT USAGE, SELECT ON ALL SEQUENCES IN SCHEMA industria TO pbl_app;

-- 3. Concessão granular por tabela
GRANT SELECT, INSERT ON industria.auditoria TO pbl_app;
GRANT SELECT, INSERT ON industria.movimentacoes TO pbl_app;
GRANT SELECT, INSERT, UPDATE ON industria.estoques TO pbl_app;
GRANT SELECT, UPDATE ON industria.componentes TO pbl_app;
GRANT SELECT ON industria.depositos TO pbl_app;
GRANT SELECT ON industria.equipamentos TO pbl_app;
GRANT SELECT ON industria.fornecedores TO pbl_app;
GRANT SELECT, INSERT, UPDATE ON industria.importacoes TO pbl_app;
GRANT SELECT, INSERT ON industria.itens_ordem TO pbl_app;
GRANT SELECT, INSERT, UPDATE ON industria.ordens_servico TO pbl_app;
GRANT SELECT, INSERT ON industria.servicos_ordem TO pbl_app;
GRANT SELECT ON industria.tecnicos TO pbl_app;
"""

with conectar_testes() as conn:
    with conn.cursor() as cur:
        cur.execute(sql_revisao_acessos)
        conn.commit()
        print("Plano de permissões mínimas aplicado com sucesso no banco de testes!")

# Ficha técnica | INC-007

## 1. Hipótese e resultado esperado
A TI suspeitou de concessão de permissões desnecessárias e de alto risco para a conta da aplicação (`pbl_app`).
- **Hipótese:** A concessão indiscriminada de `CREATE ON SCHEMA` e `ALL PRIVILEGES` viola o Princípio do Menor Privilégio, possibilitando comandos DDL e DML destrutivos (`TRUNCATE`, `DROP`).
- **Resultado esperado:** A aplicação deve possuir apenas os privilégios de leitura e escrita estritamente indispensáveis para sua operação, com proibição explícita de comandos DDL (`CREATE`) e comandos destrutivos (`TRUNCATE`, `DROP`, `DELETE` em histórico).

## 2. Procedimento de investigação
1. Leitura e análise do script de provisionamento `configuracao/00_configurar_acesso.sql`.
2. Consulta à view `information_schema.table_privileges` para catalogar todos os privilégios da role `pbl_app`.
3. Teste de execução de comandos DDL (`CREATE TABLE`) com o usuário da aplicação.
4. Mapeamento das operações executadas por cada módulo Python (`operacoes.py`, `importacao.py`, `relatorios.py`).

## 3. Evidências e resultados
- Confirmou-se a concessão de privilégios como `TRUNCATE`, `DELETE`, `TRIGGER` e `CREATE` sobre todas as 12 tabelas da base.
- Demonstrou-se que `pbl_app` conseguia criar e excluir tabelas diretamente no schema de produção/homologação.
- O plano de permissões mínimas reduziu o escopo de privilégios em mais de 60%, preservando a integridade das rotinas operacionais legítimas.

## 4. Conclusão
Incidente plenamente confirmado e diagnosticado. O perfil de acesso inicial configurava grave risco de segurança e integridade operacional.

## 5. Solução proposta e validação
1. Aplicar o script de revogação de `CREATE ON SCHEMA` e `ALL PRIVILEGES`.
2. Conceder permissões granulares por tabela e revogar `TRUNCATE` e `DELETE` em histórico e auditoria.
3. Testar todas as opções do menu do `app.py` garantindo ausência de erros de permissão em rotinas normais, e certificar que comandos de `TRUNCATE` retornem `permission denied`.